# Flipkart Gridlock 2.0 — Bengaluru Traffic Demand Prediction
**Objective:** Maximize R² (Score = max(0, 100 × R²))

**Strategy:** CatBoostRegressor with geohash×timestamp target encoding as the dominant signal. Validated using a timestamp-aligned holdout that mirrors the test set distribution.

## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostRegressor
from sklearn.metrics import r2_score

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

print('Libraries loaded successfully.')

Libraries loaded successfully.


## 2. Load Data

In [2]:
# ── Adjust paths if needed ───────────────────────────────────────────────────
TRAIN_PATH  = 'train.csv'
TEST_PATH   = 'test.csv'
SUB_PATH    = 'sample_submission.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sub   = pd.read_csv(SUB_PATH)

print(f'Train shape : {train.shape}')
print(f'Test  shape : {test.shape}')
print(f'Sample sub  : {sub.shape}')
print('\nTrain columns:', train.columns.tolist())
print('\nTrain dtypes:\n', train.dtypes)
print('\nMissing values (train):\n', train.isnull().sum())
print('\nMissing values (test):\n',  test.isnull().sum())

Train shape : (77299, 11)
Test  shape : (41778, 10)
Sample sub  : (5, 2)

Train columns: ['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather']

Train dtypes:
 Index              int64
geohash              str
day                int64
timestamp            str
demand           float64
RoadType             str
NumberofLanes      int64
LargeVehicles        str
Landmarks            str
Temperature      float64
Weather              str
dtype: object

Missing values (train):
 Index               0
geohash             0
day                 0
timestamp           0
demand              0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797
dtype: int64

Missing values (test):
 Index               0
geohash             0
day                 0
timestamp           0
RoadType          324
NumberofLanes       0
LargeVehicles       0
Landmarks 

## 3. EDA Snapshot

In [3]:
print('=== TARGET DISTRIBUTION ===')
print(train['demand'].describe())

print('\n=== KEY CARDINALITIES ===')
for col in ['geohash','day','timestamp','RoadType','NumberofLanes','LargeVehicles','Landmarks','Weather']:
    print(f'  {col}: {train[col].nunique()} unique')

# Critical insight: test is day=49 only; train day 49 covers only first ~2h of night
print('\n=== DAY DISTRIBUTION ===')
print(train['day'].value_counts())
print('Test day:', test['day'].unique())

print('\n=== DEMAND BY ROADTYPE ===')
print(train.groupby('RoadType')['demand'].agg(['mean','std','count']).round(4))

# Test timestamp window: 2:15 – 13:45  (47 of 96 possible 15-min slots)
def ts_to_min(t):
    h, m = t.split(':')
    return int(h)*60 + int(m)

test_ts_set = set(test['timestamp'].unique())
print(f'\nTest timestamps ({len(test_ts_set)}): {sorted(test_ts_set, key=ts_to_min)[:5]} … {sorted(test_ts_set, key=ts_to_min)[-3:]}')

# Geohash coverage
train_geo = set(train['geohash'])
test_geo  = set(test['geohash'])
print(f'\nGeohash coverage: {len(test_geo & train_geo)}/{len(test_geo)} test geohashes seen in train')

# geo×ts coverage
train_pairs = set(zip(train['geohash'], train['timestamp']))
test_pairs  = set(zip(test['geohash'],  test['timestamp']))
print(f'geo×timestamp coverage: {len(test_pairs & train_pairs)}/{len(test_pairs)} pairs seen in train')

=== TARGET DISTRIBUTION ===
count    7.729900e+04
mean     9.394238e-02
std      1.421905e-01
min      6.245650e-07
25%      1.822723e-02
50%      4.775994e-02
75%      1.085951e-01
max      1.000000e+00
Name: demand, dtype: float64

=== KEY CARDINALITIES ===
  geohash: 1249 unique
  day: 2 unique
  timestamp: 96 unique
  RoadType: 3 unique
  NumberofLanes: 5 unique
  LargeVehicles: 2 unique
  Landmarks: 2 unique
  Weather: 4 unique

=== DAY DISTRIBUTION ===
day
48    69427
49     7872
Name: count, dtype: int64
Test day: [49]

=== DEMAND BY ROADTYPE ===
               mean     std  count
RoadType                          
Highway      0.6108  0.2294   3560
Residential  0.0572  0.0521  69230
Street       0.2732  0.0367   3909

Test timestamps (47): ['2:15', '2:30', '2:45', '3:0', '3:15'] … ['13:15', '13:30', '13:45']

Geohash coverage: 1180/1190 test geohashes seen in train
geo×timestamp coverage: 37136/41778 pairs seen in train


## 4. Validation Strategy

**Design rationale:**
- Test set = day 49, timestamps 2:15–13:45 only.
- Train day 49 covers only timestamps 0:00–2:00 → **not representative** of test.
- Best proxy: day 48 rows **with test-matching timestamps** as the holdout.
- Train fold: day 48 rows outside that window + all of day 49.
- Target encodings computed strictly from the train fold to avoid leakage.

In [6]:
def ts_to_min(t):
    h, m = t.split(':')
    return int(h)*60 + int(m)

TEST_TS_SET = set(test['timestamp'].unique())   # 47 timestamps matching test window

train48 = train[train['day'] == 48].copy()
train49 = train[train['day'] == 49].copy()

val_df = train48[train48['timestamp'].isin(TEST_TS_SET)].copy().reset_index(drop=True)
tr_df  = pd.concat([
    train48[~train48['timestamp'].isin(TEST_TS_SET)],
    train49
], ignore_index=True)

print(f'Train fold : {len(tr_df):,} rows')
print(f'Val fold   : {len(val_df):,} rows  (mirrors test timestamp window)')
print(f'Val geo coverage: {val_df["geohash"].nunique()} / {train["geohash"].nunique()} geohashes')

Train fold : 35,448 rows
Val fold   : 41,851 rows  (mirrors test timestamp window)
Val geo coverage: 1224 / 1249 geohashes


## 5. Feature Engineering

In [7]:
def build_features(df: pd.DataFrame, ref_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build all features for df using ref_df as the encoding source.
    For validation: ref_df = tr_df
    For test:       ref_df = full train
    This prevents target leakage.
    """
    df = df.copy()
    global_mean = ref_df['demand'].mean()

    # ── 5a. Timestamp features ────────────────────────────────────────────────
    df['ts_min']     = df['timestamp'].apply(ts_to_min)
    df['hour']       = df['ts_min'] // 60
    df['minute_slot']= (df['ts_min'] % 60) // 15
    df['is_rush_am'] = ((df['hour'] >= 7) & (df['hour'] <= 9)).astype(int)
    df['is_rush_pm'] = ((df['hour'] >= 17) & (df['hour'] <= 20)).astype(int)
    df['is_night']   = ((df['hour'] >= 23) | (df['hour'] <= 5)).astype(int)
    df['sin_hour']   = np.sin(2 * np.pi * df['ts_min'] / 1440)
    df['cos_hour']   = np.cos(2 * np.pi * df['ts_min'] / 1440)

    # ── 5b. Geohash prefix features ───────────────────────────────────────────
    df['geo4'] = df['geohash'].str[:4]   # neighbourhood-level (~4.5km²)
    df['geo5'] = df['geohash'].str[:5]   # block-level (~0.56km²)

    # ── 5c. Road type imputation + ordinal ────────────────────────────────────
    rt_order   = {'Residential': 0, 'Street': 1, 'Highway': 2}
    geo_rt_mode = (
        ref_df.groupby('geohash')['RoadType']
              .agg(lambda x: x.dropna().mode()[0] if not x.dropna().empty else 'Residential')
              .to_dict()
    )
    df['road_type_filled'] = df['RoadType'].copy()
    rt_na = df['road_type_filled'].isna()
    df.loc[rt_na, 'road_type_filled'] = df.loc[rt_na, 'geohash'].map(geo_rt_mode)
    df['road_type_filled'] = df['road_type_filled'].fillna('Residential')
    df['road_type_ord']    = df['road_type_filled'].map(rt_order).fillna(0).astype(int)

    # ── 5d. Binary flags & interactions ──────────────────────────────────────
    df['large_veh_bin'] = (df['LargeVehicles'] == 'Allowed').astype(int)
    df['landmark_bin']  = (df['Landmarks'] == 'Yes').astype(int)
    df['lanes_x_road']  = df['NumberofLanes'] * df['road_type_ord']

    # ── 5e. Temperature imputation ────────────────────────────────────────────
    weather_temp_mean = ref_df.groupby('Weather')['Temperature'].mean().to_dict()
    global_temp       = ref_df['Temperature'].mean()
    df['temp_filled'] = df['Temperature'].copy()
    t_na = df['temp_filled'].isna()
    df.loc[t_na, 'temp_filled'] = df.loc[t_na, 'Weather'].map(weather_temp_mean)
    df['temp_filled']  = df['temp_filled'].fillna(global_temp)
    df['weather_filled'] = df['Weather'].fillna('Sunny')

    # ── 5f. Target encodings (computed from ref_df only) ─────────────────────
    geo_mean       = ref_df.groupby('geohash')['demand'].mean().to_dict()
    geo4_mean      = ref_df.groupby(ref_df['geohash'].str[:4])['demand'].mean().to_dict()
    geo_ts_dict    = ref_df.groupby(['geohash','timestamp'])['demand'].mean().to_dict()

    ref_h          = ref_df.copy()
    ref_h['hour']  = ref_h['timestamp'].apply(ts_to_min) // 60
    geo_hour_dict  = ref_h.groupby(['geohash','hour'])['demand'].mean().to_dict()
    rt_ts_dict     = ref_df.groupby(['RoadType','timestamp'])['demand'].mean().to_dict()
    rt_hour_dict   = ref_h.groupby(['RoadType','hour'])['demand'].mean().to_dict()
    ts_mean_dict   = ref_df.groupby('timestamp')['demand'].mean().to_dict()
    geo_std_dict   = ref_df.groupby('geohash')['demand'].std().fillna(0).to_dict()
    geo_count_dict = ref_df.groupby('geohash')['demand'].count().to_dict()

    df['geo_mean']      = df['geohash'].map(geo_mean).fillna(global_mean)
    df['geo4_mean']     = df['geo4'].map(geo4_mean).fillna(global_mean)

    # geo×timestamp — the dominant feature (~47% importance)
    df['geo_ts_mean']   = [geo_ts_dict.get((g, ts), np.nan)
                           for g, ts in zip(df['geohash'], df['timestamp'])]
    df['geo_ts_mean']   = df['geo_ts_mean'].fillna(df['geo_mean'])

    # geo×hour — fallback when exact timestamp unseen
    df['geo_hour_mean'] = [geo_hour_dict.get((g, h), np.nan)
                           for g, h in zip(df['geohash'], df['hour'])]
    df['geo_hour_mean'] = df['geo_hour_mean'].fillna(df['geo_mean'])

    df['rt_ts_mean']    = [rt_ts_dict.get((rt, ts), np.nan)
                           for rt, ts in zip(df['road_type_filled'], df['timestamp'])]
    df['rt_ts_mean']    = df['rt_ts_mean'].fillna(global_mean)

    df['rt_hour_mean']  = [rt_hour_dict.get((rt, h), np.nan)
                           for rt, h in zip(df['road_type_filled'], df['hour'])]
    df['rt_hour_mean']  = df['rt_hour_mean'].fillna(global_mean)

    df['ts_mean']       = df['timestamp'].map(ts_mean_dict).fillna(global_mean)
    df['geo_std']       = df['geohash'].map(geo_std_dict).fillna(0)
    df['geo_count']     = df['geohash'].map(geo_count_dict).fillna(0)

    # Delta: how much demand deviates at this time vs the geohash average
    df['geo_ts_delta']  = df['geo_ts_mean'] - df['geo_mean']

    return df

print('Feature engineering function defined.')
print('Building train fold features...')
tr_fe  = build_features(tr_df, tr_df)
print('Building val fold features...')
val_fe = build_features(val_df, tr_df)
print('Done. Shape:', tr_fe.shape)

Feature engineering function defined.
Building train fold features...
Building val fold features...
Done. Shape: (35448, 38)


## 6. Train CatBoost — Validation Fold

In [8]:
FEATURES = [
    # Time
    'ts_min', 'hour', 'minute_slot', 'is_rush_am', 'is_rush_pm', 'is_night',
    'sin_hour', 'cos_hour',
    # Road / infrastructure
    'NumberofLanes', 'large_veh_bin', 'landmark_bin', 'lanes_x_road',
    'temp_filled', 'road_type_ord',
    # Target encodings
    'geo_mean', 'geo4_mean', 'geo_ts_mean', 'geo_hour_mean',
    'rt_ts_mean', 'rt_hour_mean', 'ts_mean', 'geo_std', 'geo_count',
    'geo_ts_delta',
    # Categoricals handled natively by CatBoost
    'geohash', 'geo4', 'road_type_filled', 'weather_filled'
]

CAT_FEATURES = ['geohash', 'geo4', 'road_type_filled', 'weather_filled']

X_tr  = tr_fe[FEATURES]
y_tr  = tr_fe['demand']
X_val = val_fe[FEATURES]
y_val = val_fe['demand']

print(f'Training on {len(X_tr):,} rows, validating on {len(X_val):,} rows.')
print(f'Features: {len(FEATURES)} ({len(CAT_FEATURES)} categorical)')

Training on 35,448 rows, validating on 41,851 rows.
Features: 28 (4 categorical)


In [9]:
model_val = CatBoostRegressor(
    iterations          = 5000,
    learning_rate       = 0.05,
    depth               = 6,
    l2_leaf_reg         = 5,
    min_data_in_leaf    = 20,
    bagging_temperature = 1.0,
    random_strength     = 1.0,
    cat_features        = CAT_FEATURES,
    eval_metric         = 'R2',
    loss_function       = 'RMSE',
    verbose             = 200,
    random_seed         = SEED,
    early_stopping_rounds = 200
)

model_val.fit(
    X_tr, y_tr,
    eval_set      = (X_val, y_val),
    use_best_model= True
)

0:	learn: 0.0871516	test: -0.0146263	best: -0.0146263 (0)	total: 64.1ms	remaining: 5m 20s
200:	learn: 0.9688934	test: 0.5861392	best: 0.5899058 (142)	total: 1.61s	remaining: 38.4s
400:	learn: 0.9727621	test: 0.5964105	best: 0.5965778 (397)	total: 2.9s	remaining: 33.3s
600:	learn: 0.9750748	test: 0.5980458	best: 0.5989109 (429)	total: 4.16s	remaining: 30.4s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.598910925
bestIteration = 429

Shrink model to first 430 iterations.


CatBoostRegressor(bagging_temperature=1.0, cat_features=['geohash', 'geo4', 'road_type_filled', 'weather_filled'], depth=6, early_stopping_rounds=200, eval_metric='R2', iterations=5000, l2_leaf_reg=5, learning_rate=0.05, loss_function='RMSE', min_data_in_leaf=20, random_seed=42, random_strength=1.0, verbose=200)

## 7. Validation Score & Feature Importance

In [10]:
val_preds = model_val.predict(X_val)
r2_val    = r2_score(y_val, val_preds)

print('=' * 50)
print(f'  LOCAL VALIDATION R²   : {r2_val:.6f}')
print(f'  COMPETITION SCORE EST : {max(0, 100 * r2_val):.4f}')
print(f'  Best iteration        : {model_val.best_iteration_}')
print('=' * 50)

  LOCAL VALIDATION R²   : 0.598911
  COMPETITION SCORE EST : 59.8911
  Best iteration        : 429


In [11]:
fi = pd.Series(
    model_val.get_feature_importance(),
    index=FEATURES
).sort_values(ascending=False)

print('=== FEATURE IMPORTANCE (top 20) ===')
print(fi.head(20).round(4).to_string())

=== FEATURE IMPORTANCE (top 20) ===
geo_ts_mean         47.5442
geo_hour_mean       11.4424
rt_ts_mean           8.6792
lanes_x_road         7.1735
road_type_filled     5.7274
rt_hour_mean         4.3427
large_veh_bin        3.3422
road_type_ord        3.1974
geo_ts_delta         1.5971
ts_min               1.3445
hour                 0.9770
sin_hour             0.8023
geo_std              0.7934
geo_mean             0.7259
geo4                 0.5721
geo4_mean            0.4820
NumberofLanes        0.4434
temp_filled          0.3494
geohash              0.1659
weather_filled       0.0914


## 8. Train on Full Dataset

In [12]:
print('Building full-train features...')
full_fe = build_features(train, train)
X_full  = full_fe[FEATURES]
y_full  = full_fe['demand']

# Use best iteration from validation × 1.10 (more data → slightly more iterations)
FULL_ITERS = int(model_val.best_iteration_ * 1.10)
print(f'Full-train iterations: {FULL_ITERS}')

model_full = CatBoostRegressor(
    iterations          = FULL_ITERS,
    learning_rate       = 0.05,
    depth               = 6,
    l2_leaf_reg         = 5,
    min_data_in_leaf    = 20,
    bagging_temperature = 1.0,
    random_strength     = 1.0,
    cat_features        = CAT_FEATURES,
    loss_function       = 'RMSE',
    verbose             = 100,
    random_seed         = SEED
)

model_full.fit(X_full, y_full)
print('Full-train model trained.')

Building full-train features...
Full-train iterations: 471
0:	learn: 0.1355560	total: 11.3ms	remaining: 5.29s
100:	learn: 0.0165567	total: 987ms	remaining: 3.62s
200:	learn: 0.0152256	total: 1.93s	remaining: 2.59s
300:	learn: 0.0147135	total: 2.83s	remaining: 1.59s
400:	learn: 0.0143244	total: 3.73s	remaining: 652ms
470:	learn: 0.0140885	total: 4.38s	remaining: 0us
Full-train model trained.


## 9. Predict Test & Generate Submission

In [13]:
print('Building test features...')
test_fe   = build_features(test, train)
X_test    = test_fe[FEATURES]

preds_test = model_full.predict(X_test)

# Demand is in [0, 1] — clip any floating-point violations
preds_test = np.clip(preds_test, 0.0, 1.0)

print(f'Prediction stats: min={preds_test.min():.5f}  mean={preds_test.mean():.5f}  max={preds_test.max():.5f}')
print(f'Negative predictions clipped: {(model_full.predict(X_test) < 0).sum()}')

Building test features...
Prediction stats: min=0.00063  mean=0.11107  max=1.00000
Negative predictions clipped: 0


In [14]:
submission = pd.DataFrame({
    'Index' : test['Index'],
    'demand': preds_test
})

submission.to_csv('submission.csv', index=False)

print('submission.csv written successfully.')
print(f'Shape: {submission.shape}')
print(submission.head(10).to_string())

submission.csv written successfully.
Shape: (41778, 2)
   Index    demand
0      0  0.040932
1      1  0.032316
2      2  0.026169
3      3  0.073864
4      4  0.108551
5      5  0.013427
6      6  0.021081
7      7  0.318005
8      8  0.023396
9      9  0.073096


## 10. Summary

| Item | Value |
|------|-------|
| Local R² | See cell output above |
| Est. competition score | max(0, 100 × R²) |
| Dominant feature | `geo_ts_mean` (geo×timestamp target encoding) |
| Validation design | Timestamp-aligned day-48 holdout |
| Model | CatBoostRegressor depth=6, lr=0.05 |